# BD-TrafficGuard — Robust Bangladeshi Traffic Sign Detection & Recognition

This notebook implements the project requirements from the supplied proposal and presentation:

- Bangladesh Road Traffic Sign Dataset: detection with 31 classes
- clean train/validation/test split
- YOLO baseline
- degradation-aware robust YOLO
- controlled corruption benchmark: Gaussian blur, motion blur, low light, glare, fog, rain, occlusion, JPEG compression, small-object/resolution degradation
- clean vs corrupted evaluation
- mAP@0.5, mAP@0.5:0.95, precision, recall, F1
- robustness drop
- class-wise analysis
- inference time / FPS / model size
- ablation experiments
- qualitative failure analysis
- optional Faster R-CNN comparison
- optional two-stage detector + classifier using EfficientNet-B0
- export for lightweight deployment

**Important:** set `DATASET_SOURCE` in Cell 2 to your downloaded dataset location. The notebook is designed to work with a YOLO-format dataset or with a dataset that already contains image/annotation folders that can be converted.

In [ ]:
# ============================================================
# 1. KAGGLE ENVIRONMENT SETUP
# ============================================================
# This notebook is designed for Kaggle. Ultralytics is not
# guaranteed to be pre-installed, so install it before importing YOLO.
# IMPORTANT: In Kaggle, turn ON: Settings -> Internet -> On

import sys, subprocess, importlib.util

if importlib.util.find_spec("ultralytics") is None:
    print("Ultralytics not found. Installing...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--disable-pip-version-check", "ultralytics"
    ])
else:
    print("Ultralytics is already installed.")

import ultralytics
print("Ultralytics version:", ultralytics.__version__)

# Confirm that the Kaggle GPU is visible.
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Enable a Kaggle GPU accelerator before training.")


In [ ]:
# ============================================================
# 2. FINAL PROJECT CONFIGURATION — YOLO11s ONLY
# ============================================================

from pathlib import Path
import yaml, os, random, time, shutil, math
from collections import Counter
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO  # installed/verified in Cell 1

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

KAGGLE_INPUT = Path("/kaggle/input")
yaml_candidates = list(KAGGLE_INPUT.rglob("data.yaml"))
if not yaml_candidates:
    raise FileNotFoundError("No data.yaml found under /kaggle/input/. Attach the Bangladesh traffic-sign dataset.")
DATA_YAML = yaml_candidates[0]
DATASET_ROOT = DATA_YAML.parent

with open(DATA_YAML, "r", encoding="utf-8") as f:
    DATA_CONFIG = yaml.safe_load(f)

CLASS_NAMES = DATA_CONFIG.get("names")
NUM_CLASSES = DATA_CONFIG.get("nc", len(CLASS_NAMES) if CLASS_NAMES else None)
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES, key=lambda x: int(x))]
if not CLASS_NAMES:
    raise ValueError("Class names missing from data.yaml.")

TRAIN_IMAGES, TRAIN_LABELS = DATASET_ROOT/"train/images", DATASET_ROOT/"train/labels"
VAL_IMAGES, VAL_LABELS = DATASET_ROOT/"valid/images", DATASET_ROOT/"valid/labels"
TEST_IMAGES, TEST_LABELS = DATASET_ROOT/"test/images", DATASET_ROOT/"test/labels"

IMAGE_EXTENSIONS = {".jpg",".jpeg",".png",".bmp",".webp"}

PROJECT_ROOT = Path("/kaggle/working/BD-TrafficGuard")
RUNS_ROOT = PROJECT_ROOT/"runs"
CORRUPTION_ROOT = PROJECT_ROOT/"BD-TrafficSign-C"
RESULTS_ROOT = PROJECT_ROOT/"results"
MODELS_ROOT = PROJECT_ROOT/"models"
PLOTS_ROOT = PROJECT_ROOT/"plots"

for d in [PROJECT_ROOT,RUNS_ROOT,CORRUPTION_ROOT,RESULTS_ROOT,MODELS_ROOT,PLOTS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 640
BATCH_SIZE = 16
WORKERS = 2
EPOCHS = 50
DEVICE = 0 if torch.cuda.is_available() else "cpu"

BASELINE_MODEL_NAME = "YOLO11s"
PROPOSED_MODEL_NAME = "BD-TrafficGuard"
YOLO_WEIGHTS = "yolo11s.pt"

print("Dataset:", DATASET_ROOT)
print("Classes:", NUM_CLASSES)
print("YOLO architecture:", YOLO_WEIGHTS)
print("Image size:", IMG_SIZE, "| Epochs:", EPOCHS, "| Batch:", BATCH_SIZE)
print("Device:", DEVICE)

In [ ]:
# ============================================================
# 3. DATASET VALIDATION & CLASS DISTRIBUTION
# ============================================================

from collections import Counter

print("=" * 70)
print("BD-TRAFFICGUARD DATASET VALIDATION")
print("=" * 70)


# ============================================================
# 1. CHECK IMAGE ↔ LABEL MATCHING
# ============================================================

def get_image_stem_set(directory):
    return {
        p.stem
        for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    }


def get_label_stem_set(directory):
    return {
        p.stem
        for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() == ".txt"
    }


splits = {
    "train": (TRAIN_IMAGES, TRAIN_LABELS),
    "valid": (VAL_IMAGES, VAL_LABELS),
    "test": (TEST_IMAGES, TEST_LABELS),
}

matching_summary = {}

for split_name, (image_dir, label_dir) in splits.items():

    image_stems = get_image_stem_set(image_dir)
    label_stems = get_label_stem_set(label_dir)

    missing_labels = image_stems - label_stems
    orphan_labels = label_stems - image_stems

    matching_summary[split_name] = {
        "images": len(image_stems),
        "labels": len(label_stems),
        "missing_labels": len(missing_labels),
        "orphan_labels": len(orphan_labels),
    }

    print(f"\n[{split_name.upper()}]")
    print(f"Images          : {len(image_stems):,}")
    print(f"Labels          : {len(label_stems):,}")
    print(f"Missing labels  : {len(missing_labels):,}")
    print(f"Orphan labels   : {len(orphan_labels):,}")

    if missing_labels:
        print("  Example missing labels:", list(missing_labels)[:5])

    if orphan_labels:
        print("  Example orphan labels:", list(orphan_labels)[:5])


# ============================================================
# 2. VALIDATE YOLO LABEL FORMAT
# ============================================================

print("\n" + "=" * 70)
print("YOLO LABEL VALIDATION")
print("=" * 70)

invalid_files = []
invalid_rows = []

total_objects = 0

for split_name, (_, label_dir) in splits.items():

    label_files = list(label_dir.glob("*.txt"))

    for label_file in tqdm(
        label_files,
        desc=f"Checking {split_name} labels"
    ):

        try:
            with open(label_file, "r") as f:
                lines = [line.strip() for line in f if line.strip()]

            for line_number, line in enumerate(lines, start=1):

                parts = line.split()

                # YOLO detection format:
                # class_id x_center y_center width height
                if len(parts) != 5:
                    invalid_rows.append({
                        "file": str(label_file),
                        "line": line_number,
                        "reason": "Expected 5 values",
                        "content": line,
                    })
                    continue

                try:
                    class_id = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except ValueError:
                    invalid_rows.append({
                        "file": str(label_file),
                        "line": line_number,
                        "reason": "Non-numeric value",
                        "content": line,
                    })
                    continue

                # Class ID check
                if not (0 <= class_id < NUM_CLASSES):
                    invalid_rows.append({
                        "file": str(label_file),
                        "line": line_number,
                        "reason": f"Invalid class ID: {class_id}",
                        "content": line,
                    })
                    continue

                # Bounding box values must be normalized [0,1]
                bbox_values = [x, y, w, h]

                if not all(0 <= v <= 1 for v in bbox_values):
                    invalid_rows.append({
                        "file": str(label_file),
                        "line": line_number,
                        "reason": "Bounding box outside [0,1]",
                        "content": line,
                    })
                    continue

                # Width and height must be > 0
                if w <= 0 or h <= 0:
                    invalid_rows.append({
                        "file": str(label_file),
                        "line": line_number,
                        "reason": "Zero/negative width or height",
                        "content": line,
                    })
                    continue

                total_objects += 1

        except Exception as e:

            invalid_files.append({
                "file": str(label_file),
                "error": str(e),
            })


print("\nTotal annotated objects:", f"{total_objects:,}")
print("Invalid label rows      :", f"{len(invalid_rows):,}")
print("Unreadable label files  :", f"{len(invalid_files):,}")


# ============================================================
# 3. CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

class_counts = {
    "train": Counter(),
    "valid": Counter(),
    "test": Counter(),
}

object_counts_per_split = {}

for split_name, (_, label_dir) in splits.items():

    counter = Counter()

    for label_file in tqdm(
        label_dir.glob("*.txt"),
        desc=f"Counting {split_name} classes"
    ):

        with open(label_file, "r") as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                if len(parts) != 5:
                    continue

                try:
                    class_id = int(parts[0])
                except ValueError:
                    continue

                if 0 <= class_id < NUM_CLASSES:
                    counter[class_id] += 1

    class_counts[split_name] = counter

    object_counts_per_split[split_name] = sum(counter.values())


# ============================================================
# 4. CREATE DISTRIBUTION TABLE
# ============================================================

distribution_df = pd.DataFrame({
    "class_id": range(NUM_CLASSES),
    "class_name": CLASS_NAMES,
    "train": [
        class_counts["train"][i]
        for i in range(NUM_CLASSES)
    ],
    "valid": [
        class_counts["valid"][i]
        for i in range(NUM_CLASSES)
    ],
    "test": [
        class_counts["test"][i]
        for i in range(NUM_CLASSES)
    ],
})

distribution_df["total"] = (
    distribution_df["train"]
    + distribution_df["valid"]
    + distribution_df["test"]
)

distribution_df["train_pct"] = (
    distribution_df["train"]
    / distribution_df["train"].sum()
    * 100
)

distribution_df["total_pct"] = (
    distribution_df["total"]
    / distribution_df["total"].sum()
    * 100
)

display(distribution_df)


# ============================================================
# 5. SUMMARY STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

for split_name in ["train", "valid", "test"]:

    num_images = matching_summary[split_name]["images"]
    num_objects = object_counts_per_split[split_name]

    print(
        f"{split_name.capitalize():6s}: "
        f"{num_images:,} images | "
        f"{num_objects:,} annotated objects"
    )


print("\nMost represented classes:")

display(
    distribution_df[
        ["class_id", "class_name", "total"]
    ]
    .sort_values("total", ascending=False)
    .head(10)
)


print("\nLeast represented classes:")

display(
    distribution_df[
        ["class_id", "class_name", "total"]
    ]
    .sort_values("total", ascending=True)
    .head(10)
)


# ============================================================
# 6. PLOT CLASS DISTRIBUTION
# ============================================================

plt.figure(figsize=(16, 8))

plot_df = distribution_df.sort_values(
    "total",
    ascending=True
)

plt.barh(
    plot_df["class_name"],
    plot_df["total"]
)

plt.xlabel("Number of annotated objects")
plt.ylabel("Traffic-sign class")
plt.title("BD-TrafficGuard — Overall Class Distribution")

plt.tight_layout()
plt.show()


# ============================================================
# 7. SAVE DISTRIBUTION
# ============================================================

distribution_path = (
    TABLES_ROOT / "class_distribution.csv"
)

distribution_df.to_csv(
    distribution_path,
    index=False
)

print("\nSaved:")
print(distribution_path)


# ============================================================
# 8. FINAL VALIDATION STATUS
# ============================================================

print("\n" + "=" * 70)

if (
    len(invalid_rows) == 0
    and len(invalid_files) == 0
    and all(
        x["missing_labels"] == 0
        for x in matching_summary.values()
    )
):

    print("✓ DATASET VALIDATION PASSED")

else:

    print("⚠ DATASET VALIDATION FOUND ISSUES")

    if invalid_rows:
        print(
            f"  Invalid annotation rows: "
            f"{len(invalid_rows)}"
        )

    if invalid_files:
        print(
            f"  Unreadable label files: "
            f"{len(invalid_files)}"
        )

print("=" * 70)

In [ ]:
# ============================================================
# 4. VISUAL DATASET INSPECTION
# ============================================================

import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIGURATION
# ============================================================

NUM_SAMPLES = 12

random.seed(SEED)


# ============================================================
# HELPER: DRAW YOLO ANNOTATIONS
# ============================================================

def load_yolo_annotations(label_path):
    """
    Read YOLO-format annotations.

    Format:
        class_id x_center y_center width height

    Coordinates are normalized to [0, 1].
    """

    annotations = []

    if not label_path.exists():
        return annotations

    with open(label_path, "r") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center, y_center, width, height = map(
                float,
                parts[1:]
            )

            annotations.append(
                (
                    class_id,
                    x_center,
                    y_center,
                    width,
                    height
                )
            )

    return annotations


def draw_yolo_boxes(image, annotations):
    """
    Draw YOLO bounding boxes and class names.
    """

    image = image.copy()

    height, width = image.shape[:2]

    for (
        class_id,
        x_center,
        y_center,
        box_width,
        box_height
    ) in annotations:

        # Convert normalized coordinates
        # to pixel coordinates.

        x1 = int(
            (x_center - box_width / 2) * width
        )

        y1 = int(
            (y_center - box_height / 2) * height
        )

        x2 = int(
            (x_center + box_width / 2) * width
        )

        y2 = int(
            (y_center + box_height / 2) * height
        )

        # Keep coordinates inside image.
        x1 = max(0, min(x1, width - 1))
        y1 = max(0, min(y1, height - 1))
        x2 = max(0, min(x2, width - 1))
        y2 = max(0, min(y2, height - 1))

        class_name = CLASS_NAMES[class_id]

        # Draw rectangle
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Label
        label = f"{class_id}: {class_name}"

        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.45
        thickness = 1

        (text_width, text_height), baseline = cv2.getTextSize(
            label,
            font,
            font_scale,
            thickness
        )

        text_y = max(
            y1,
            text_height + baseline + 2
        )

        # Background for text
        cv2.rectangle(
            image,
            (x1, text_y - text_height - baseline - 2),
            (
                x1 + text_width + 4,
                text_y + 2
            ),
            (0, 255, 0),
            -1
        )

        # Text
        cv2.putText(
            image,
            label,
            (x1 + 2, text_y - 2),
            font,
            font_scale,
            (0, 0, 0),
            thickness,
            cv2.LINE_AA
        )

    return image


# ============================================================
# GET RANDOM TRAINING IMAGES
# ============================================================

train_images = [
    p
    for p in TRAIN_IMAGES.iterdir()
    if p.is_file()
    and p.suffix.lower() in IMAGE_EXTENSIONS
]

if len(train_images) < NUM_SAMPLES:
    NUM_SAMPLES = len(train_images)

sample_images = random.sample(
    train_images,
    NUM_SAMPLES
)


# ============================================================
# DISPLAY
# ============================================================

fig, axes = plt.subplots(
    3,
    4,
    figsize=(20, 15)
)

axes = axes.flatten()

for ax, image_path in zip(
    axes,
    sample_images
):

    # Read image
    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        ax.axis("off")
        continue

    # Convert BGR → RGB
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    # Corresponding label
    label_path = TRAIN_LABELS / (
        image_path.stem + ".txt"
    )

    annotations = load_yolo_annotations(
        label_path
    )

    # Draw boxes
    annotated_image = draw_yolo_boxes(
        image,
        annotations
    )

    ax.imshow(annotated_image)

    ax.set_title(
        f"{image_path.name}\n"
        f"{len(annotations)} object(s)"
    )

    ax.axis("off")


# Hide unused axes
for ax in axes[len(sample_images):]:
    ax.axis("off")


plt.suptitle(
    "BD-TrafficGuard — Training Dataset Annotation Inspection",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. DATASET STATISTICS & OBJECT-SIZE ANALYSIS
# ============================================================

from collections import Counter
from PIL import Image

print("=" * 70)
print("BD-TRAFFICGUARD DATASET STATISTICS")
print("=" * 70)


# ============================================================
# CONFIGURATION
# ============================================================

MAX_IMAGES_FOR_DIMENSION_SCAN = None
# None = scan all images
#
# If this is too slow, you can use:
# MAX_IMAGES_FOR_DIMENSION_SCAN = 2000


# ============================================================
# 1. IMAGE DIMENSIONS
# ============================================================

print("\n" + "=" * 70)
print("IMAGE DIMENSIONS")
print("=" * 70)


def collect_image_dimensions(image_dir, max_images=None):

    image_files = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if max_images is not None:
        image_files = image_files[:max_images]

    dimensions = []

    for image_path in tqdm(
        image_files,
        desc=f"Scanning {image_dir.name}"
    ):

        try:
            with Image.open(image_path) as img:
                width, height = img.size

            dimensions.append({
                "file": image_path.name,
                "width": width,
                "height": height,
                "aspect_ratio": width / height
            })

        except Exception as e:
            print(
                f"Could not read {image_path}: {e}"
            )

    return pd.DataFrame(dimensions)


dimension_dfs = {}

for split_name, image_dir in [
    ("train", TRAIN_IMAGES),
    ("valid", VAL_IMAGES),
    ("test", TEST_IMAGES)
]:

    dimension_dfs[split_name] = collect_image_dimensions(
        image_dir,
        MAX_IMAGES_FOR_DIMENSION_SCAN
    )


# ============================================================
# DIMENSION SUMMARY
# ============================================================

dimension_summary = []

for split_name, df in dimension_dfs.items():

    dimension_summary.append({
        "split": split_name,
        "images": len(df),
        "unique_widths": df["width"].nunique(),
        "unique_heights": df["height"].nunique(),
        "mean_width": df["width"].mean(),
        "mean_height": df["height"].mean(),
        "min_width": df["width"].min(),
        "max_width": df["width"].max(),
        "min_height": df["height"].min(),
        "max_height": df["height"].max(),
        "mean_aspect_ratio": df["aspect_ratio"].mean()
    })


dimension_summary_df = pd.DataFrame(
    dimension_summary
)

display(dimension_summary_df)


# ============================================================
# MOST COMMON IMAGE RESOLUTIONS
# ============================================================

all_dimensions = pd.concat(
    dimension_dfs.values(),
    ignore_index=True
)

resolution_counts = (
    all_dimensions
    .groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nMost common image resolutions:")

display(
    resolution_counts.head(15)
)


# ============================================================
# 2. OBJECT STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("BOUNDING-BOX STATISTICS")
print("=" * 70)


object_records = []

for split_name, (image_dir, label_dir) in splits.items():

    image_files = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for image_path in tqdm(
        image_files,
        desc=f"Analyzing {split_name} objects"
    ):

        label_path = label_dir / (
            image_path.stem + ".txt"
        )

        if not label_path.exists():
            continue

        # Read image dimensions
        try:
            with Image.open(image_path) as img:
                image_width, image_height = img.size
        except:
            continue

        with open(label_path, "r") as f:
            lines = [
                line.strip()
                for line in f
                if line.strip()
            ]

        for line in lines:

            parts = line.split()

            if len(parts) != 5:
                continue

            try:
                class_id = int(parts[0])
                xc, yc, bw, bh = map(
                    float,
                    parts[1:]
                )
            except:
                continue

            # Convert normalized bbox to pixel dimensions
            box_width = bw * image_width
            box_height = bh * image_height

            box_area = (
                box_width *
                box_height
            )

            image_area = (
                image_width *
                image_height
            )

            area_ratio = (
                box_area /
                image_area
            )

            object_records.append({

                "split": split_name,

                "class_id": class_id,

                "class_name": (
                    CLASS_NAMES[class_id]
                    if 0 <= class_id < NUM_CLASSES
                    else "Unknown"
                ),

                "image_width": image_width,

                "image_height": image_height,

                "box_width": box_width,

                "box_height": box_height,

                "box_area": box_area,

                "box_area_ratio": area_ratio,

                "box_aspect_ratio": (
                    box_width / box_height
                    if box_height > 0
                    else np.nan
                )
            })


objects_df = pd.DataFrame(
    object_records
)

print(
    "Total objects analyzed:",
    f"{len(objects_df):,}"
)


# ============================================================
# 3. OBJECT SIZE SUMMARY
# ============================================================

object_summary = (
    objects_df
    .groupby("split")
    .agg(
        objects=("class_id", "count"),

        mean_box_width=("box_width", "mean"),
        median_box_width=("box_width", "median"),

        mean_box_height=("box_height", "mean"),
        median_box_height=("box_height", "median"),

        mean_area_ratio=("box_area_ratio", "mean"),
        median_area_ratio=("box_area_ratio", "median"),

        mean_aspect_ratio=("box_aspect_ratio", "mean"),
        median_aspect_ratio=("box_aspect_ratio", "median")
    )
    .reset_index()
)

display(object_summary)


# ============================================================
# 4. SMALL OBJECT ANALYSIS
# ============================================================
# We define object size by bounding-box area relative to
# the complete image.
#
# These thresholds are descriptive, not training rules.

SMALL_THRESHOLD = 0.01
MEDIUM_THRESHOLD = 0.05

objects_df["size_category"] = pd.cut(
    objects_df["box_area_ratio"],
    bins=[
        -np.inf,
        SMALL_THRESHOLD,
        MEDIUM_THRESHOLD,
        np.inf
    ],
    labels=[
        "Small (<1%)",
        "Medium (1%-5%)",
        "Large (>5%)"
    ]
)


size_distribution = (
    objects_df["size_category"]
    .value_counts()
    .reindex([
        "Small (<1%)",
        "Medium (1%-5%)",
        "Large (>5%)"
    ])
    .fillna(0)
    .astype(int)
    .reset_index()
)

size_distribution.columns = [
    "size_category",
    "objects"
]

size_distribution["percentage"] = (
    size_distribution["objects"]
    / size_distribution["objects"].sum()
    * 100
)

print("\nObject-size distribution:")

display(size_distribution)


# ============================================================
# 5. OBJECTS PER IMAGE
# ============================================================

print("\n" + "=" * 70)
print("OBJECTS PER IMAGE")
print("=" * 70)


objects_per_image = []

for split_name, (_, label_dir) in splits.items():

    for label_path in label_dir.glob("*.txt"):

        try:
            with open(label_path, "r") as f:
                count = sum(
                    1
                    for line in f
                    if line.strip()
                )

            objects_per_image.append({
                "split": split_name,
                "image": label_path.stem,
                "objects": count
            })

        except:
            pass


objects_per_image_df = pd.DataFrame(
    objects_per_image
)


objects_per_image_summary = (
    objects_per_image_df
    .groupby("split")
    ["objects"]
    .agg([
        "count",
        "mean",
        "median",
        "min",
        "max"
    ])
    .reset_index()
)

display(objects_per_image_summary)


# ============================================================
# 6. OBJECTS PER IMAGE DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

for split_name in ["train", "valid", "test"]:

    values = objects_per_image_df[
        objects_per_image_df["split"] == split_name
    ]["objects"]

    plt.hist(
        values,
        bins=range(
            int(values.min()),
            int(values.max()) + 2
        ),
        alpha=0.5,
        label=split_name
    )

plt.xlabel("Number of objects per image")
plt.ylabel("Number of images")
plt.title(
    "Objects per Image Distribution"
)
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 7. OBJECT SIZE DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    objects_df["box_area_ratio"],
    bins=50
)

plt.axvline(
    SMALL_THRESHOLD,
    linestyle="--",
    label="1% image area"
)

plt.axvline(
    MEDIUM_THRESHOLD,
    linestyle="--",
    label="5% image area"
)

plt.xlabel(
    "Bounding-box area / image area"
)

plt.ylabel("Number of objects")

plt.title(
    "Traffic Sign Relative Object-Size Distribution"
)

plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 8. BOX WIDTH VS HEIGHT
# ============================================================

plt.figure(figsize=(9, 7))

plt.scatter(
    objects_df["box_width"],
    objects_df["box_height"],
    alpha=0.25,
    s=8
)

plt.xlabel("Bounding-box width (pixels)")
plt.ylabel("Bounding-box height (pixels)")

plt.title(
    "Traffic Sign Bounding-Box Dimensions"
)

plt.tight_layout()
plt.show()


# ============================================================
# 9. SAVE STATISTICS
# ============================================================

objects_df.to_csv(
    TABLES_ROOT / "object_statistics.csv",
    index=False
)

dimension_summary_df.to_csv(
    TABLES_ROOT / "image_dimension_summary.csv",
    index=False
)

resolution_counts.to_csv(
    TABLES_ROOT / "image_resolution_distribution.csv",
    index=False
)

object_summary.to_csv(
    TABLES_ROOT / "object_size_summary.csv",
    index=False
)

size_distribution.to_csv(
    TABLES_ROOT / "object_size_distribution.csv",
    index=False
)

objects_per_image_summary.to_csv(
    TABLES_ROOT / "objects_per_image_summary.csv",
    index=False
)


print("\n" + "=" * 70)
print("STATISTICS SAVED")
print("=" * 70)

print(
    "Output directory:",
    TABLES_ROOT
)

## 7. Final six-condition Bangladesh-specific corruption benchmark

The main experiment is deliberately limited to:

**Blur · Rain · Fog · Glare · Low Light · Occlusion**

Each has mild, moderate, and severe levels.

The removed conditions (Gaussian noise, motion blur, JPEG compression, low contrast) are not part of the final reported benchmark.

In [ ]:
# ============================================================
# 7. CORRUPTION FUNCTIONS
# ============================================================

CORRUPTION_LEVELS = {"mild": 1, "moderate": 2, "severe": 3}
CORRUPTIONS = ["blur", "rain", "fog", "glare", "low_light", "occlusion"]

def load_yolo_annotations(label_path):
    annotations = []
    if not label_path.exists():
        return annotations
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        try:
            cid = int(parts[0])
            xc, yc, w, h = map(float, parts[1:])
            annotations.append((cid, xc, yc, w, h))
        except:
            pass
    return annotations

def draw_yolo_boxes(image, annotations):
    out = image.copy()
    h, w = out.shape[:2]
    for cid, xc, yc, bw, bh in annotations:
        x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
        x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
        x1, y1 = max(0,x1), max(0,y1)
        x2, y2 = min(w-1,x2), min(h-1,y2)
        cv2.rectangle(out,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(out, CLASS_NAMES[cid], (x1,max(15,y1-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, .45, (0,255,0), 1, cv2.LINE_AA)
    return out

def blur(image, severity):
    return cv2.GaussianBlur(image, ({1:3,2:7,3:13}[severity],)*2, 0)

def rain(image, severity, seed_offset=0):
    rng = np.random.default_rng(SEED + severity*1000 + seed_offset)
    out = image.copy()
    overlay = np.zeros_like(out)
    h,w = out.shape[:2]
    count = {1:80,2:160,3:260}[severity]
    length = {1:10,2:18,3:28}[severity]
    for _ in range(count):
        x = int(rng.integers(0,w)); y = int(rng.integers(0,h))
        cv2.line(overlay,(x,y),(min(w-1,x+4),min(h-1,y+length)),(210,210,210),1)
    return cv2.addWeighted(out,1.0,overlay,{1:.20,2:.35,3:.50}[severity],0)

def fog(image, severity):
    strength = {1:.15,2:.30,3:.50}[severity]
    h,w = image.shape[:2]
    y = np.linspace(0,1,h).reshape(-1,1,1)
    transmission = 1.0 - strength*(1-y)
    atm = np.full_like(image,255,dtype=np.float32)
    result = image.astype(np.float32)*transmission + atm*(1-transmission)
    return np.clip(result,0,255).astype(np.uint8)

def glare(image, severity):
    out = image.astype(np.float32).copy()
    h,w = out.shape[:2]
    strength = {1:.30,2:.55,3:.85}[severity]
    rx = int(w*{1:.12,2:.20,3:.30}[severity])
    ry = int(h*{1:.12,2:.20,3:.30}[severity])
    cx,cy = int(w*.75),int(h*.20)
    yy,xx = np.mgrid[0:h,0:w]
    dist = ((xx-cx)/max(rx,1))**2 + ((yy-cy)/max(ry,1))**2
    mask = np.clip(1-dist,0,1)**2
    out += mask[...,None]*255*strength
    out = cv2.GaussianBlur(out,(0,0),sigmaX=max(1,3*severity))
    return np.clip(out,0,255).astype(np.uint8)

def low_light(image, severity):
    gamma = {1:.75,2:.50,3:.30}[severity]
    normalized = image.astype(np.float32)/255
    return np.clip((normalized**(1/gamma))*255,0,255).astype(np.uint8)

def occlusion(image, annotations, severity, seed_offset=0):
    """Cover part of an actual traffic-sign bounding box."""
    out = image.copy()
    if not annotations:
        return out
    rng = np.random.default_rng(SEED + severity*100 + seed_offset)
    cid,xc,yc,bw,bh = annotations[int(rng.integers(0,len(annotations)))]
    h,w = out.shape[:2]
    x1=max(0,int((xc-bw/2)*w)); y1=max(0,int((yc-bh/2)*h))
    x2=min(w-1,int((xc+bw/2)*w)); y2=min(h-1,int((yc+bh/2)*h))
    coverage={1:.25,2:.40,3:.60}[severity]
    ow=max(1,int(max(2,x2-x1)*np.sqrt(coverage)))
    oh=max(1,int(max(2,y2-y1)*np.sqrt(coverage)))
    ox1=int(rng.integers(x1,max(x1+1,x2-ow+1)))
    oy1=int(rng.integers(y1,max(y1+1,y2-oh+1)))
    ox2=min(x2,ox1+ow); oy2=min(y2,oy1+oh)
    cv2.rectangle(out,(ox1,oy1),(ox2,oy2),(55,55,55),-1)
    return out

def apply_corruption(image, annotations, name, severity, seed_offset=0):
    if name=="blur": return blur(image,severity)
    if name=="rain": return rain(image,severity,seed_offset)
    if name=="fog": return fog(image,severity)
    if name=="glare": return glare(image,severity)
    if name=="low_light": return low_light(image,severity)
    if name=="occlusion": return occlusion(image,annotations,severity,seed_offset)
    raise ValueError(name)

print("Final benchmark:", CORRUPTIONS)

In [ ]:
# ============================================================
# 8. ONE VISUAL AUGMENTATION DEMONSTRATION
# ============================================================

train_paths = sorted([p for p in TRAIN_IMAGES.iterdir()
                      if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

demo_path = None
demo_annotations = []
for p in random.sample(train_paths, min(100,len(train_paths))):
    anns = load_yolo_annotations(TRAIN_LABELS/f"{p.stem}.txt")
    if anns:
        demo_path, demo_annotations = p, anns
        break

if demo_path is None:
    raise RuntimeError("No demonstration image with annotations was found.")

original = cv2.imread(str(demo_path))

fig, axes = plt.subplots(6,2,figsize=(12,24))
for row,name in enumerate(CORRUPTIONS):
    transformed = apply_corruption(
        original.copy(), demo_annotations, name, severity=2, seed_offset=row
    )
    axes[row,0].imshow(cv2.cvtColor(draw_yolo_boxes(original,demo_annotations),cv2.COLOR_BGR2RGB))
    axes[row,0].set_title(f"Original → {name.replace('_',' ').title()}")
    axes[row,0].axis("off")
    axes[row,1].imshow(cv2.cvtColor(draw_yolo_boxes(transformed,demo_annotations),cv2.COLOR_BGR2RGB))
    axes[row,1].set_title(f"After {name.replace('_',' ').title()}")
    axes[row,1].axis("off")

plt.suptitle("BD-TrafficGuard: Visual augmentation demonstration",fontsize=18)
plt.tight_layout()
plt.show()

print("Demo image:", demo_path.name)
print("Severity shown: moderate")

In [ ]:
# ============================================================
# 9. CREATE BD-TRAFFICSIGN-C — TEST ONLY
# ============================================================

def create_eval_yaml(condition, severity):
    condition_dir = CORRUPTION_ROOT/condition/severity
    yaml_path = CORRUPTION_ROOT/f"{condition}_{severity}.yaml"
    cfg = {
        "path": str(condition_dir),
        "train": "images",
        "val": "images",
        "test": "images",
        "nc": NUM_CLASSES,
        "names": CLASS_NAMES,
    }
    with open(yaml_path,"w",encoding="utf-8") as f:
        yaml.safe_dump(cfg,f,sort_keys=False,allow_unicode=True)
    return yaml_path

def generate_corruption_benchmark():
    for p in list(CORRUPTION_ROOT.iterdir()):
        if p.is_dir():
            shutil.rmtree(p)

    test_files = sorted([p for p in TEST_IMAGES.iterdir()
                         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

    for name in CORRUPTIONS:
        for sev_name,sev in CORRUPTION_LEVELS.items():
            out_i = CORRUPTION_ROOT/name/sev_name/"images"
            out_l = CORRUPTION_ROOT/name/sev_name/"labels"
            out_i.mkdir(parents=True,exist_ok=True)
            out_l.mkdir(parents=True,exist_ok=True)

            for p in tqdm(test_files,desc=f"{name}/{sev_name}"):
                image = cv2.imread(str(p))
                if image is None: continue
                anns = load_yolo_annotations(TEST_LABELS/f"{p.stem}.txt")
                transformed = apply_corruption(
                    image,anns,name,sev,seed_offset=__import__('zlib').crc32(p.name.encode())%100000
                )
                cv2.imwrite(str(out_i/p.name),transformed)
                label = TEST_LABELS/f"{p.stem}.txt"
                if label.exists(): shutil.copy2(label,out_l/label.name)

            create_eval_yaml(name,sev_name)

generate_corruption_benchmark()

print("Created",len(CORRUPTIONS)*len(CORRUPTION_LEVELS),"conditions:",
      len(CORRUPTIONS),"corruptions ×",len(CORRUPTION_LEVELS),"severities.")

## 10. Train the YOLO11s standard baseline

The baseline uses normal YOLO11s training on the original clean dataset.

In [ ]:
# ============================================================
# 10. YOLO11s STANDARD BASELINE TRAINING
# ============================================================

BASELINE_PROJECT = RUNS_ROOT/"baseline"
BASELINE_RUN_NAME = "yolo11s_standard"

model = YOLO(YOLO_WEIGHTS)
start = time.time()

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(BASELINE_PROJECT),
    name=BASELINE_RUN_NAME,
    pretrained=True,
    seed=SEED,
    deterministic=True,
    patience=15,
    plots=True,
    cache=False,
    verbose=True,
    amp=True,
)

baseline_training_time = time.time()-start
BASELINE_BEST = BASELINE_PROJECT/BASELINE_RUN_NAME/"weights"/"best.pt"
if not BASELINE_BEST.exists():
    raise FileNotFoundError(BASELINE_BEST)

CENTRAL_BASELINE = MODELS_ROOT/"YOLO11s_baseline_best.pt"
shutil.copy2(BASELINE_BEST,CENTRAL_BASELINE)

print("Training time:",f"{baseline_training_time/60:.2f} min")
print("Checkpoint:",CENTRAL_BASELINE)

In [ ]:
# ============================================================
# 11. BASELINE CLEAN TEST EVALUATION
# ============================================================

def extract_metrics(results):
    p=float(results.box.mp); r=float(results.box.mr)
    return {
        "precision":p,
        "recall":r,
        "F1":2*p*r/max(p+r,1e-12),
        "mAP50":float(results.box.map50),
        "mAP50_95":float(results.box.map),
    }

baseline_eval = YOLO(str(CENTRAL_BASELINE))
clean_results = baseline_eval.val(
    data=str(DATA_YAML),split="test",imgsz=IMG_SIZE,batch=BATCH_SIZE,
    workers=WORKERS,device=DEVICE,plots=True,save_json=False,verbose=True
)
baseline_clean = extract_metrics(clean_results)
display(pd.DataFrame([{"model":BASELINE_MODEL_NAME,"condition":"clean",**baseline_clean}]).round(4))

## 12. Build the BD-TrafficGuard training dataset

The proposed model keeps YOLO11s unchanged and changes the **training data**. Training images receive explicit degradation-aware transformations, plus a small/distant road-scene simulation. Validation and test remain clean.

In [ ]:
# ============================================================
# 12. SMALL / DISTANT SIGN SIMULATION
# ============================================================

def scale_down_scene(image, annotations, factor):
    h,w=image.shape[:2]
    nw,nh=max(1,int(w*factor)),max(1,int(h*factor))
    resized=cv2.resize(image,(nw,nh),interpolation=cv2.INTER_AREA)
    canvas=np.full_like(image,128)
    ox,oy=(w-nw)//2,(h-nh)//2
    canvas[oy:oy+nh,ox:ox+nw]=resized

    updated=[]
    for cid,xc,yc,bw,bh in annotations:
        updated.append((
            cid,
            np.clip((ox+factor*xc*w)/w,0,1),
            np.clip((oy+factor*yc*h)/h,0,1),
            np.clip(factor*bw,1e-6,1),
            np.clip(factor*bh,1e-6,1)
        ))
    return canvas,updated

small_img,small_anns=scale_down_scene(original,demo_annotations,.65)
fig,ax=plt.subplots(1,2,figsize=(14,6))
ax[0].imshow(cv2.cvtColor(draw_yolo_boxes(original,demo_annotations),cv2.COLOR_BGR2RGB))
ax[0].set_title("Original road scene"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(draw_yolo_boxes(small_img,small_anns),cv2.COLOR_BGR2RGB))
ax[1].set_title("Small/distant sign simulation"); ax[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 13. BUILD EXPLICIT DEGRADATION-AWARE TRAINING DATASET
# ============================================================

ROBUST_DATASET_ROOT=PROJECT_ROOT/"datasets"/"bd_trafficguard_train"
if ROBUST_DATASET_ROOT.exists():
    shutil.rmtree(ROBUST_DATASET_ROOT)

def save_yolo_labels(path,annotations):
    with open(path,"w",encoding="utf-8") as f:
        for cid,xc,yc,bw,bh in annotations:
            f.write(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

manifest=[]
split_map={"train":(TRAIN_IMAGES,TRAIN_LABELS),
           "valid":(VAL_IMAGES,VAL_LABELS),
           "test":(TEST_IMAGES,TEST_LABELS)}

for split,(img_dir,label_dir) in split_map.items():
    out_i=ROBUST_DATASET_ROOT/split/"images"
    out_l=ROBUST_DATASET_ROOT/split/"labels"
    out_i.mkdir(parents=True,exist_ok=True); out_l.mkdir(parents=True,exist_ok=True)

    files=sorted([p for p in img_dir.iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])
    rng=random.Random(SEED+{"train":0,"valid":1000,"test":2000}[split])

    for p in tqdm(files,desc=f"Building {split}"):
        image=cv2.imread(str(p))
        if image is None: continue
        anns=load_yolo_annotations(label_dir/f"{p.stem}.txt")
        applied=[]

        if split=="train":
            # Explicit small/distant augmentation.
            if rng.random()<0.35:
                factor=rng.uniform(.55,.85)
                image,anns=scale_down_scene(image,anns,factor)
                applied.append(f"small_distance_{factor:.2f}")

            # Explicit Bangladesh-specific degradation.
            if rng.random()<0.75:
                name=rng.choice(CORRUPTIONS)
                sev_name=rng.choice(list(CORRUPTION_LEVELS))
                image=apply_corruption(
                    image,anns,name,CORRUPTION_LEVELS[sev_name],
                    seed_offset=__import__('zlib').crc32(p.name.encode())%100000
                )
                applied.append(f"{name}_{sev_name}")

        cv2.imwrite(str(out_i/p.name),image)
        label_out=out_l/f"{p.stem}.txt"
        save_yolo_labels(label_out,anns)
        manifest.append({"split":split,"image":p.name,
                         "applied":"|".join(applied) if applied else "clean"})

manifest_df=pd.DataFrame(manifest)
manifest_df.to_csv(RESULTS_ROOT/"robust_training_manifest.csv",index=False)

ROBUST_DATA_YAML=ROBUST_DATASET_ROOT/"data.yaml"
robust_cfg={"path":str(ROBUST_DATASET_ROOT),
            "train":"train/images","val":"valid/images","test":"test/images",
            "nc":NUM_CLASSES,"names":CLASS_NAMES}
with open(ROBUST_DATA_YAML,"w",encoding="utf-8") as f:
    yaml.safe_dump(robust_cfg,f,sort_keys=False,allow_unicode=True)

print("Training transformations:")
display(manifest_df[manifest_df.split=="train"].applied.value_counts().head(20))

## 14. Train BD-TrafficGuard

**BD-TrafficGuard = YOLO11s + explicit degradation-aware training + small/distant sign-focused augmentation.**

There is only one proposed robust model.

In [ ]:
# ============================================================
# 14. BD-TRAFFICGUARD TRAINING
# ============================================================

PROPOSED_PROJECT=RUNS_ROOT/"proposed"
PROPOSED_RUN_NAME="bd_trafficguard_yolo11s"

proposed=YOLO(YOLO_WEIGHTS)
start=time.time()

proposed.train(
    data=str(ROBUST_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROPOSED_PROJECT),
    name=PROPOSED_RUN_NAME,
    pretrained=True,
    seed=SEED,
    deterministic=True,
    patience=15,
    plots=True,
    cache=False,
    verbose=True,
    amp=True,

    # Conservative geometry/scale augmentation complements
    # the explicit corruption dataset.
    degrees=5.0,
    translate=0.10,
    scale=0.50,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.50,
    mixup=0.0,
)

proposed_training_time=time.time()-start
PROPOSED_BEST=PROPOSED_PROJECT/PROPOSED_RUN_NAME/"weights"/"best.pt"
if not PROPOSED_BEST.exists():
    raise FileNotFoundError(PROPOSED_BEST)

CENTRAL_PROPOSED=MODELS_ROOT/"BD-TrafficGuard_best.pt"
shutil.copy2(PROPOSED_BEST,CENTRAL_PROPOSED)

print("Training time:",f"{proposed_training_time/60:.2f} min")
print("Checkpoint:",CENTRAL_PROPOSED)

## 15. Evaluate both final models

Both models are evaluated on exactly the same clean test set and the same 18 corrupted test conditions (6 corruptions × 3 severities).

In [ ]:
# ============================================================
# 15. CLEAN + CORRUPTED EVALUATION
# ============================================================

def evaluate_model(model_path,model_name):
    model=YOLO(str(model_path))
    rows=[]

    clean=model.val(data=str(DATA_YAML),split="test",imgsz=IMG_SIZE,
                    batch=BATCH_SIZE,workers=WORKERS,device=DEVICE,
                    verbose=False,plots=False,save_json=False)
    m=extract_metrics(clean)
    rows.append({"model":model_name,"condition":"clean","severity":"clean",**m})

    for name in CORRUPTIONS:
        for sev_name in CORRUPTION_LEVELS:
            yml=create_eval_yaml(name,sev_name)
            result=model.val(data=str(yml),split="test",imgsz=IMG_SIZE,
                             batch=BATCH_SIZE,workers=WORKERS,device=DEVICE,
                             verbose=False,plots=False,save_json=False)
            m=extract_metrics(result)
            rows.append({"model":model_name,"condition":name,
                         "severity":sev_name,**m})

    df=pd.DataFrame(rows)
    clean50=float(df.loc[df.condition=="clean","mAP50"].iloc[0])
    clean95=float(df.loc[df.condition=="clean","mAP50_95"].iloc[0])
    df["mAP50_drop"]=clean50-df["mAP50"]
    df["mAP50_95_drop"]=clean95-df["mAP50_95"]
    df["mAP50_drop_pct"]=df["mAP50_drop"]/max(clean50,1e-12)*100
    df["mAP50_95_drop_pct"]=df["mAP50_95_drop"]/max(clean95,1e-12)*100
    return df

baseline_results_df=evaluate_model(CENTRAL_BASELINE,BASELINE_MODEL_NAME)
proposed_results_df=evaluate_model(CENTRAL_PROPOSED,PROPOSED_MODEL_NAME)
all_results_df=pd.concat([baseline_results_df,proposed_results_df],ignore_index=True)

display(all_results_df.round(4))

## 16. Main comparison and Robustness Drop

**Robustness Drop = Clean mAP − Corrupted mAP**. Smaller drop means more of the clean performance is retained under degradation.

In [ ]:
# ============================================================
# 16. SUMMARY
# ============================================================

def summarize(group):
    clean=group[group.condition=="clean"].iloc[0]
    bad=group[group.condition!="clean"]
    return pd.Series({
        "clean_precision":clean.precision,
        "clean_recall":clean.recall,
        "clean_F1":clean.F1,
        "clean_mAP50":clean.mAP50,
        "clean_mAP50_95":clean.mAP50_95,
        "mean_corrupted_mAP50":bad.mAP50.mean(),
        "mean_corrupted_mAP50_95":bad.mAP50_95.mean(),
        "worst_corrupted_mAP50_95":bad.mAP50_95.min(),
        "mean_robustness_drop_mAP50_95":bad.mAP50_95_drop.mean(),
        "mean_robustness_drop_pct":bad.mAP50_95_drop_pct.mean(),
    })

summary_df=all_results_df.groupby("model",group_keys=False).apply(summarize).reset_index()
display(summary_df.round(4))

condition_summary=(
    all_results_df[all_results_df.condition!="clean"]
    .groupby(["condition","model"])
    .agg(mean_mAP50=("mAP50","mean"),
         mean_mAP50_95=("mAP50_95","mean"),
         mean_precision=("precision","mean"),
         mean_recall=("recall","mean"),
         mean_F1=("F1","mean"),
         mean_drop=("mAP50_95_drop","mean"))
    .reset_index()
)
display(condition_summary.round(4))

all_results_df.to_csv(RESULTS_ROOT/"results.csv",index=False)
summary_df.to_csv(RESULTS_ROOT/"model_summary.csv",index=False)
condition_summary.to_csv(RESULTS_ROOT/"condition_summary.csv",index=False)

In [ ]:
# ============================================================
# 17. ROBUSTNESS PLOTS
# ============================================================

plt.figure(figsize=(12,6))
for model_name in [BASELINE_MODEL_NAME,PROPOSED_MODEL_NAME]:
    s=condition_summary[condition_summary.model==model_name]
    plt.plot(s.condition,s.mean_mAP50_95,marker="o",label=model_name)
plt.ylim(0,1)
plt.ylabel("Mean mAP@0.5:0.95")
plt.xlabel("Adverse condition")
plt.title("YOLO11s vs BD-TrafficGuard")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_ROOT/"condition_robustness_comparison.png",dpi=180,bbox_inches="tight")
plt.show()

plt.figure(figsize=(12,6))
for model_name in [BASELINE_MODEL_NAME,PROPOSED_MODEL_NAME]:
    s=condition_summary[condition_summary.model==model_name]
    plt.plot(s.condition,s.mean_drop,marker="o",label=model_name)
plt.ylabel("Mean robustness drop (mAP@0.5:0.95)")
plt.xlabel("Adverse condition")
plt.title("Robustness Drop by adverse condition")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_ROOT/"robustness_drop_comparison.png",dpi=180,bbox_inches="tight")
plt.show()

## 18. Class-wise clean performance

Keep the class-wise analysis focused on the two final models. Note that classes with no test examples cannot support meaningful held-out test metrics.

In [ ]:
# ============================================================
# 18. CLASS-WISE CLEAN AP
# ============================================================

def classwise(model_path,model_name):
    model=YOLO(str(model_path))
    r=model.val(data=str(DATA_YAML),split="test",imgsz=IMG_SIZE,
                batch=BATCH_SIZE,workers=WORKERS,device=DEVICE,
                verbose=False,plots=False,save_json=False)
    ap50=np.asarray(r.box.ap50).reshape(-1)
    ap=np.asarray(r.box.ap).reshape(-1)
    return pd.DataFrame([{
        "model":model_name,"class_id":cid,"class_name":CLASS_NAMES[cid],
        "AP50":float(ap50[cid]) if cid<len(ap50) else np.nan,
        "AP50_95":float(ap[cid]) if cid<len(ap) else np.nan
    } for cid in range(NUM_CLASSES)])

classwise_df=pd.concat([
    classwise(CENTRAL_BASELINE,BASELINE_MODEL_NAME),
    classwise(CENTRAL_PROPOSED,PROPOSED_MODEL_NAME)
],ignore_index=True)

display(classwise_df.round(4))
classwise_df.to_csv(RESULTS_ROOT/"classwise_clean_performance.csv",index=False)

## 19. Qualitative comparison

Show representative clean and severe-condition predictions. For the final report, use actual error cases where possible rather than only random successful examples.

In [ ]:
# ============================================================
# 19. QUALITATIVE PREDICTIONS
# ============================================================

def show_prediction_grid(model_path,model_name,condition="clean",severity="severe",n=6):
    model=YOLO(str(model_path))
    if condition=="clean":
        paths=sorted([p for p in TEST_IMAGES.iterdir()
                      if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])
    else:
        d=CORRUPTION_ROOT/condition/severity/"images"
        paths=sorted([p for p in d.iterdir()
                      if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

    selected=random.sample(paths,min(n,len(paths)))
    fig,axes=plt.subplots(2,3,figsize=(16,10))
    axes=np.asarray(axes).reshape(-1)

    for ax,p in zip(axes,selected):
        r=model.predict(source=str(p),imgsz=IMG_SIZE,conf=.25,
                         device=DEVICE,verbose=False)[0]
        ax.imshow(cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB))
        ax.set_title(p.name); ax.axis("off")
    for ax in axes[len(selected):]: ax.axis("off")

    plt.suptitle(f"{model_name} — {condition.replace('_',' ').title()} / {severity}",fontsize=16)
    plt.tight_layout(); plt.show()

# Run these selectively if runtime is limited.
for condition in CORRUPTIONS:
    show_prediction_grid(CENTRAL_BASELINE,BASELINE_MODEL_NAME,condition,"severe",6)
    show_prediction_grid(CENTRAL_PROPOSED,PROPOSED_MODEL_NAME,condition,"severe",6)

## 20. Deployment metrics

Measure model size, average inference time, and FPS on the available hardware. Do not generalize a GPU result to phones or edge devices.

In [ ]:
# ============================================================
# 20. FPS / INFERENCE TIME / MODEL SIZE
# ============================================================

def model_size_mb(path):
    return Path(path).stat().st_size/(1024**2)

def benchmark(model_path,n=50):
    paths=sorted([p for p in TEST_IMAGES.iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])[:n]
    model=YOLO(str(model_path))

    for p in paths[:min(5,len(paths))]:
        model.predict(source=str(p),imgsz=IMG_SIZE,device=DEVICE,verbose=False)

    times=[]
    for p in tqdm(paths,desc=f"Timing {Path(model_path).name}"):
        t=time.perf_counter()
        model.predict(source=str(p),imgsz=IMG_SIZE,device=DEVICE,verbose=False)
        times.append(time.perf_counter()-t)

    avg=float(np.mean(times))
    return {"model_size_MB":model_size_mb(model_path),
            "avg_ms_per_image":avg*1000,
            "FPS":1/max(avg,1e-12),
            "hardware":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "images_timed":len(paths)}

deployment_df=pd.DataFrame([
    {"model":BASELINE_MODEL_NAME,**benchmark(CENTRAL_BASELINE)},
    {"model":PROPOSED_MODEL_NAME,**benchmark(CENTRAL_PROPOSED)}
])
display(deployment_df.round(4))
deployment_df.to_csv(RESULTS_ROOT/"deployment.csv",index=False)

## 21. Final project result table

This is the table to use in the report and presentation. All values are generated from the actual evaluation cells.

In [ ]:
# ============================================================
# 21. FINAL RESULT TABLE
# ============================================================

final_table=summary_df[[
    "model","clean_mAP50","clean_mAP50_95","clean_precision",
    "clean_recall","clean_F1","mean_corrupted_mAP50",
    "mean_corrupted_mAP50_95","mean_robustness_drop_pct"
]].copy()

final_table.columns=[
    "Model","Clean mAP50","Clean mAP50:95","Clean Precision",
    "Clean Recall","Clean F1","Mean Corrupted mAP50",
    "Mean Corrupted mAP50:95","Mean Robustness Drop (%)"
]
display(final_table.round(4))

presentation_table=condition_summary.pivot(
    index="condition",columns="model",values="mean_mAP50_95"
).reset_index()
display(presentation_table.round(4))

## 22. Simple export

Only `results.csv`, plots, and the two final model checkpoints are retained for the main experiment.

In [ ]:
# ============================================================
# 22. SIMPLE EXPORT
# ============================================================

required=[
    RESULTS_ROOT/"results.csv",
    RESULTS_ROOT/"model_summary.csv",
    RESULTS_ROOT/"condition_summary.csv",
    RESULTS_ROOT/"classwise_clean_performance.csv",
    RESULTS_ROOT/"deployment.csv",
    PLOTS_ROOT/"condition_robustness_comparison.png",
    PLOTS_ROOT/"robustness_drop_comparison.png",
    CENTRAL_BASELINE,
    CENTRAL_PROPOSED,
]

export_check=pd.DataFrame({
    "path":[str(p) for p in required],
    "exists":[p.exists() for p in required],
    "size_MB":[p.stat().st_size/(1024**2) if p.exists() else np.nan for p in required]
})
display(export_check)

print("\nFINAL DESIGN")
print("Baseline : YOLO11s + standard training")
print("Proposed : BD-TrafficGuard = YOLO11s + degradation-aware training + small/distant augmentation")
print("Benchmark:", ", ".join(x.replace("_"," ").title() for x in CORRUPTIONS))
print("Robustness Drop = Clean mAP - Corrupted mAP")

# 23. Presentation / viva explanation

### Why only YOLO11s?

The project goal is **traffic-sign detection + recognition in full road scenes**. YOLO already performs both tasks in one pipeline:

**Road image → YOLO11s → bounding box + class**

Using multiple detector families would add complexity without directly serving the central research question.

### What is new in BD-TrafficGuard?

Not a new backbone.

The contribution is:

**YOLO11s + degradation-aware training + small/distant augmentation + Bangladesh-specific six-condition corruption benchmark**

### What exactly is being tested?

> Can the same YOLO11s architecture retain good clean performance while becoming more robust to Blur, Rain, Fog, Glare, Low Light and Occlusion?

### Important scientific limitation

The corruption benchmark is synthetic and controlled. Therefore the experiment demonstrates robustness to the implemented degradation protocol; it does not prove robustness to every possible real-world Bangladeshi road condition.

### Final comparison logic

| Model | Architecture | Training |
|---|---|---|
| YOLO11s | YOLO11s | Standard clean training |
| BD-TrafficGuard | YOLO11s | Explicit degradation-aware + small/distant augmentation |

Because the architecture is identical, the comparison focuses on the effect of the training strategy.